# AR Reconciliation Workflow Engine - API Testing

This notebook provides comprehensive tests for all API endpoints of the AR Reconciliation Workflow Engine.

## Setup Instructions

```bash
# Install dependencies
uv sync
uv sync --extra dev

# Run database migrations
uv run alembic upgrade head

# Start the server (in a separate terminal)
uv run python main.py
```

## Pipeline Stages
1. **Ingestion** - Accept and validate input data
2. **Normalization** - Clean data, convert types
3. **Balance Compute** - Calculate outstanding amounts
4. **Reconciliation** - Calculate expected balance and differences
5. **Validation Rules** - Apply business rules
6. **Verdict Generation** - Generate final verdict
7. **Reporting** - Generate reports

In [32]:
import json
import time

import httpx

BASE_URL = "http://localhost:8000"
client = httpx.Client(base_url=BASE_URL, timeout=30)
print(f"Client configured for {BASE_URL}")

Client configured for http://localhost:8000


---
## 1. Health Check Endpoints

Verify the server is running and all dependencies are healthy.

In [33]:
# Basic health check
resp = client.get("/health")
print(f"GET /health -> Status: {resp.status_code}")
print(json.dumps(resp.json(), indent=2))

GET /health -> Status: 200
{
  "message": null,
  "status": "healthy",
  "version": "1.0.0",
  "timestamp": "2026-06-01T11:03:45.961332+05:30",
  "services": {
    "database": "ok"
  }
}


In [34]:
# Liveness probe (for Kubernetes)
resp = client.get("/health/liveness")
print(f"GET /health/liveness -> Status: {resp.status_code}")
print(json.dumps(resp.json(), indent=2))

GET /health/liveness -> Status: 200
{
  "message": null,
  "status": "alive"
}


In [35]:
# Readiness probe
resp = client.get("/health/readiness")
print(f"GET /health/readiness -> Status: {resp.status_code}")
print(json.dumps(resp.json(), indent=2))

GET /health/readiness -> Status: 200
{
  "message": null,
  "status": "ready",
  "details": {
    "database": "ok"
  }
}


In [36]:
# Detailed health check
resp = client.get("/health/detailed")
print(f"GET /health/detailed -> Status: {resp.status_code}")
print(json.dumps(resp.json(), indent=2))

GET /health/detailed -> Status: 200
{
  "message": null,
  "liveness": {
    "message": null,
    "status": "alive"
  },
  "readiness": {
    "message": null,
    "status": "ready",
    "details": {
      "database": "ok"
    }
  },
  "startup": {
    "message": null,
    "status": "started",
    "initialized": true
  },
  "version": "1.0.0",
  "timestamp": "2026-06-01T11:03:46.053922+05:30",
  "services": {
    "database": "ok"
  }
}


---
## 2. Submit Single AR Record

`POST /ar_records/submit` - Submit one AR record for processing. Returns a workflow ID immediately while processing happens in the background.

In [37]:
# Sample AR record from the CSV dataset
record = {
    "customer_id": "CUST-TEST-001",
    "customer_name": "Test Customer 001",
    "customer_balance": 3486.58,
    "invoice_total": 3892.38,
    "invoice_applied_amount": 241.57,
    "invoice_exchange_rate": 1.1371,
    "payment_total": 3316.88,
    "payment_applied_amount": 2375.22,
    "payment_exchange_rate": 0.8873,
    "credit_total": 910.95,
    "credit_applied_amount": 676.97,
    "credit_exchange_rate": 1.0237,
    "adjustment_total": 1310.49,
    "adjustment_applied_amount": 944.51,
    "adjustment_exchange_rate": 1.1211,
}

resp = client.post("/ar_records/submit", json=record)
print(f"POST /ar_records/submit -> Status: {resp.status_code}")
result = resp.json()
print(json.dumps(result, indent=2))

# Store workflow_id for later tests
workflow_id = result.get("workflow_id")
print(f"\nWorkflow ID: {workflow_id}")

POST /ar_records/submit -> Status: 201
{
  "message": "Record submitted successfully.",
  "customer_id": "CUST-TEST-001",
  "workflow_id": "d2e3f492-be37-4e8c-8091-3cf493b72dd7",
  "workflow_status": "PENDING",
  "workflow_current_stage": "INGESTION"
}

Workflow ID: d2e3f492-be37-4e8c-8091-3cf493b72dd7


---
## 3. Idempotency Test - Duplicate Submission

Submitting the same `customer_id` again should return the existing workflow, not create a new one.

In [38]:
# Submit the same record again - should be idempotent
resp = client.post("/ar_records/submit", json=record)
print(f"POST /ar_records/submit (duplicate) -> Status: {resp.status_code}")
result = resp.json()
print(json.dumps(result, indent=2))

# Verify it's the same workflow
assert result.get("workflow_id") == workflow_id, (
    "Expected same workflow ID for duplicate submission"
)
print("\n✅ Idempotency verified - same workflow ID returned")

POST /ar_records/submit (duplicate) -> Status: 201
{
  "message": "Duplicate submission detected. Returning existing workflow details. Use the update endpoint to modify customer data.",
  "customer_id": "CUST-TEST-001",
  "workflow_id": "d2e3f492-be37-4e8c-8091-3cf493b72dd7",
  "workflow_status": "RUNNING",
  "workflow_current_stage": "NORMALIZATION"
}

✅ Idempotency verified - same workflow ID returned


---
## 4. Get Workflow Status

`GET /ar_records/workflow/{id}` - Check the status and stage details of a workflow.

In [39]:
# Wait for background processing
print("Waiting for background processing...")
time.sleep(3)

resp = client.get(f"/ar_records/workflow/{workflow_id}")
print(f"GET /ar_records/workflow/{workflow_id} -> Status: {resp.status_code}")
workflow_detail = resp.json()

print(f"\nWorkflow Status: {workflow_detail['workflow']['status']}")
print(f"Current Stage: {workflow_detail['workflow']['current_stage']}")
print(f"Retry Count: {workflow_detail['workflow']['retry_count']}")
print(f"\nStage Details ({len(workflow_detail['stages'])} stages recorded):")
for stage in workflow_detail["stages"]:
    print(
        f"  - {stage['stage_name']}: {stage['status']} (retries: {stage['retry_count']})"
    )
    if stage.get("error_message"):
        print(f"    Error: {stage['error_message'][:100]}...")

Waiting for background processing...
GET /ar_records/workflow/d2e3f492-be37-4e8c-8091-3cf493b72dd7 -> Status: 200

Workflow Status: COMPLETED
Current Stage: REPORTING
Retry Count: 0

Stage Details (7 stages recorded):
  - INGESTION: COMPLETED (retries: 0)
  - NORMALIZATION: COMPLETED (retries: 0)
  - BALANCE_COMPUTE: COMPLETED (retries: 0)
  - RECONCILIATION: COMPLETED (retries: 0)
  - VALIDATION_RULES: COMPLETED (retries: 0)
  - VERDICT_GENERATION: COMPLETED (retries: 0)
  - REPORTING: COMPLETED (retries: 0)


---
## 5. List All Workflows

`GET /ar_records/workflows` - List workflows with optional status filter and pagination.

In [40]:
# List all workflows
resp = client.get("/ar_records/workflows", params={"limit": 10})
print(f"GET /ar_records/workflows -> Status: {resp.status_code}")
workflows = resp.json()
print(f"\nTotal workflows returned: {len(workflows)}")

for w in workflows[:5]:
    print(
        f"  {w['id'][:8]}... | {w['customer_id']} | {w['status']} | stage: {w['current_stage']}"
    )

GET /ar_records/workflows -> Status: 200

Total workflows returned: 1
  d2e3f492... | CUST-TEST-001 | COMPLETED | stage: REPORTING


In [41]:
# Filter by status
for status_filter in ["COMPLETED", "FAILED", "RUNNING", "PENDING"]:
    resp = client.get(
        "/ar_records/workflows", params={"status": status_filter, "limit": 5}
    )
    count = len(resp.json())
    print(f"{status_filter}: {count} workflows")

COMPLETED: 1 workflows
FAILED: 0 workflows
RUNNING: 0 workflows
PENDING: 0 workflows


---
## 6. Enhanced Workflow List (with Staleness Detection)

`GET /ar_records/workflows/enhanced` - List workflows with staleness flag and last error message.

In [42]:
resp = client.get("/ar_records/workflows/enhanced", params={"limit": 10})
print(f"GET /ar_records/workflows/enhanced -> Status: {resp.status_code}")
enhanced = resp.json()

print(f"\nEnhanced workflow listing ({len(enhanced)} results):")
for w in enhanced[:5]:
    stale_flag = " [STALE]" if w.get("is_stale") else ""
    error = f" | error: {w['last_error'][:50]}..." if w.get("last_error") else ""
    print(f"  {w['customer_id']} | {w['status']}{stale_flag}{error}")

GET /ar_records/workflows/enhanced -> Status: 200

Enhanced workflow listing (1 results):
  CUST-TEST-001 | COMPLETED


---
## 7. Dashboard Statistics

`GET /ar_records/stats` - Get aggregate statistics about all workflows.

In [43]:
resp = client.get("/ar_records/stats")
print(f"GET /ar_records/stats -> Status: {resp.status_code}")
stats = resp.json()

print("\n=== Workflow Statistics ===")
print(f"Total workflows: {stats['total_workflows']}")
print(f"Completed: {stats['completed']}")
print(f"Failed: {stats['failed']}")
print(f"Pending: {stats['pending']}")
print(f"Running: {stats['running']}")
print(f"Retrying: {stats['retrying']}")
print(f"Stale: {stats['stale']}")
print(f"\nCompletion Rate: {stats['completion_rate']}%")
print(f"Failure Rate: {stats['failure_rate']}%")

GET /ar_records/stats -> Status: 200

=== Workflow Statistics ===
Total workflows: 1
Completed: 1
Failed: 0
Pending: 0
Running: 0
Retrying: 0
Stale: 0

Completion Rate: 100.0%
Failure Rate: 0.0%


---
## 8. Resume Failed Workflow

`POST /ar_records/resume/{id}` - Resume a workflow that failed (retries from the last failed stage).

In [44]:
# Find a failed workflow to resume
resp = client.get("/ar_records/workflows", params={"status": "FAILED", "limit": 1})
failed = resp.json()

if failed:
    failed_id = failed[0]["id"]
    failed_customer = failed[0]["customer_id"]
    print(f"Found failed workflow: {failed_id}")
    print(f"Customer: {failed_customer}")
    print(f"Failed at stage: {failed[0]['current_stage']}")

    # Resume the workflow
    resp = client.post(f"/ar_records/resume/{failed_id}")
    print(f"\nPOST /ar_records/resume/{failed_id} -> Status: {resp.status_code}")
    print(json.dumps(resp.json(), indent=2))

    # Wait and check status
    time.sleep(3)
    resp = client.get(f"/ar_records/workflow/{failed_id}")
    new_status = resp.json()["workflow"]["status"]
    print(f"\nAfter resume - Status: {new_status}")
else:
    print("No failed workflows to resume.")
    print("Note: The system has a 20% random failure rate by default.")
    print("Submit more records or run bulk upload to generate some failures.")

No failed workflows to resume.
Note: The system has a 20% random failure rate by default.
Submit more records or run bulk upload to generate some failures.


---
## 9. Bulk Upload CSV

`POST /ar_records/bulk-upload` - Upload the CSV file to process multiple records at once. Each row becomes an independent workflow.

In [45]:
# Upload the CSV file for bulk processing
try:
    with open("../data/erp_export.csv", "rb") as f:
        resp = client.post(
            "/ar_records/bulk-upload", files={"file": ("erp_export.csv", f, "text/csv")}
        )

    print(f"POST /ar_records/bulk-upload -> Status: {resp.status_code}")
    bulk_result = resp.json()

    print("\n=== Bulk Upload Results ===")
    print(f"Total records: {bulk_result['total_records']}")
    print(f"Submitted: {bulk_result['submitted']}")
    print(f"Duplicates: {bulk_result['duplicates']}")
    print(f"Errors: {bulk_result['errors']}")
    print(f"\nFirst 5 workflow IDs: {bulk_result['workflow_ids'][:5]}")

    if bulk_result.get("error_details"):
        print(f"\nError details: {bulk_result['error_details'][:3]}")

except FileNotFoundError:
    print("CSV file not found. Run from the tests/ directory or adjust the path.")

POST /ar_records/bulk-upload -> Status: 200

=== Bulk Upload Results ===
Total records: 1000
Submitted: 1000
Duplicates: 0
Errors: 0

First 5 workflow IDs: ['1dea1a6a-2fe1-410c-9aa7-d240b5abeb85', '2d418b73-9bc3-46c5-968f-5dadedd4d3c6', '1017da9d-ee50-4ae2-ae0d-acb551c72bdc', '0395b622-7627-4d74-9a2c-0a759736ffe0', '083c560b-0112-4930-98fc-fa9465f57db8']


---
## 10. Wait and Check Final Stats

In [46]:
# Wait for bulk processing to complete
print("Waiting for bulk processing (10 seconds)...")
time.sleep(10)

resp = client.get("/ar_records/stats")
stats = resp.json()

print("\n=== Final Statistics After Bulk Processing ===")
print(f"Total workflows: {stats['total_workflows']}")
print(f"Completed: {stats['completed']} ({stats['completion_rate']}%)")
print(f"Failed: {stats['failed']} ({stats['failure_rate']}%)")
print(f"Still running: {stats['running']}")

Waiting for bulk processing (10 seconds)...

=== Final Statistics After Bulk Processing ===
Total workflows: 1001
Completed: 129 (12.89%)
Failed: 0 (0.0%)
Still running: 0


---
## 11. Export Results as CSV

`GET /ar_records/export` - Download completed workflow results as a CSV file.

In [47]:
resp = client.get("/ar_records/export")
print(f"GET /ar_records/export -> Status: {resp.status_code}")
print(f"Content-Type: {resp.headers.get('content-type')}")
print("\nCSV Preview (first 500 chars):")
print(resp.text[:500])

GET /ar_records/export -> Status: 200
Content-Type: text/csv; charset=utf-8

CSV Preview (first 500 chars):
workflow_id,customer_id,status,current_stage,retry_count,created_at,updated_at
1ee0e5f7-36a5-4671-99b6-36a87cfb7599,CUST-0129,COMPLETED,REPORTING,0,2026-06-01T05:33:51.124569,2026-06-01T05:34:14.346431
e5cac235-12ea-4d66-80ea-4e537900e963,CUST-0128,COMPLETED,REPORTING,1,2026-06-01T05:33:51.110749,2026-06-01T05:34:14.253017
3835f8be-7a4c-4985-83b6-a9cf563f5db9,CUST-0127,COMPLETED,REPORTING,3,2026-06-01T05:33:51.101767,2026-06-01T05:34:14.142837
2ec8f070-8d55-40c5-a0ba-6fb58aa363ea,CUST-0126,C


---
## 12. Error Handling Tests

Test that the API returns proper error responses for invalid input.

In [48]:
# Test 404 - workflow not found
resp = client.get("/ar_records/workflow/nonexistent-id-12345")
print(f"GET /ar_records/workflow/bad-id -> {resp.status_code} (expected 404)")
print(f"  Response: {resp.json()}")

GET /ar_records/workflow/bad-id -> 404 (expected 404)
  Response: {'detail': 'Workflow nonexistent-id-12345 not found'}


In [49]:
# Test 404 - resume non-existent workflow
resp = client.post("/ar_records/resume/nonexistent-id-12345")
print(f"POST /ar_records/resume/bad-id -> {resp.status_code} (expected 404)")
print(f"  Response: {resp.json()}")

POST /ar_records/resume/bad-id -> 404 (expected 404)
  Response: {'detail': 'Workflow nonexistent-id-12345 not found'}


In [50]:
# Test 400 - upload non-CSV file
import io

fake_file = io.BytesIO(b"not a csv")
resp = client.post(
    "/ar_records/bulk-upload", files={"file": ("test.txt", fake_file, "text/plain")}
)
print(f"POST /ar_records/bulk-upload with .txt -> {resp.status_code} (expected 400)")
print(f"  Response: {resp.json()}")

POST /ar_records/bulk-upload with .txt -> 400 (expected 400)
  Response: {'detail': 'File must be a CSV file'}


In [51]:
# Test 422 - missing required field
resp = client.post("/ar_records/submit", json={})
print(f"POST /ar_records/submit with empty body -> {resp.status_code} (expected 422)")
print(f"  Validation error: {resp.json()['detail'][0]['msg']}")

POST /ar_records/submit with empty body -> 422 (expected 422)
  Validation error: Field required


In [52]:
# Test invalid status filter
resp = client.get("/ar_records/workflows", params={"status": "INVALID_STATUS"})
print(
    f"GET /ar_records/workflows?status=INVALID_STATUS -> {resp.status_code} (expected 400)"
)
print(f"  Response: {resp.json()}")

GET /ar_records/workflows?status=INVALID_STATUS -> 400 (expected 400)
  Response: {'detail': "Invalid status: INVALID_STATUS. Valid: ['PENDING', 'RUNNING', 'FAILED', 'RETRYING', 'COMPLETED']"}


---
## 13. Resume All Failed Workflows

Loop through all failed workflows and attempt to resume them.

In [53]:
# Get all failed workflows
resp = client.get("/ar_records/workflows", params={"status": "FAILED", "limit": 100})
failed = resp.json()
print(f"Found {len(failed)} failed workflows. Resuming...\n")

for w in failed:
    r = client.post(f"/ar_records/resume/{w['id']}")
    result = r.json()
    print(f"  {w['customer_id']}: {result.get('message', result.get('status'))}")

# Wait and check final stats
print("\nWaiting for resume processing (10 seconds)...")
time.sleep(10)

resp = client.get("/ar_records/stats")
stats = resp.json()
print("\n=== Final Stats After Resume ===")
print(f"Completed: {stats['completed']}/{stats['total_workflows']}")
print(f"Still failed: {stats['failed']}")

Found 0 failed workflows. Resuming...


Waiting for resume processing (10 seconds)...

=== Final Stats After Resume ===
Completed: 222/1001
Still failed: 0


---
## 14. Concurrent Duplicate Submission Test

Fire multiple requests for the same customer_id at once - only one workflow should be created.

In [54]:
import asyncio

import httpx


async def submit_record_async(customer_id: str, idx: int):
    """Submit a record asynchronously."""
    async with httpx.AsyncClient(base_url=BASE_URL, timeout=30) as client:
        record = {
            "customer_id": customer_id,
            "customer_name": "Concurrent Test Customer",
            "customer_balance": 1000.00,
            "invoice_total": 1500.00,
            "invoice_applied_amount": 500.00,
            "invoice_exchange_rate": 1.0,
            "payment_total": 500.00,
            "payment_applied_amount": 500.00,
            "payment_exchange_rate": 1.0,
            "credit_total": 0.0,
            "credit_applied_amount": 0.0,
            "credit_exchange_rate": 1.0,
            "adjustment_total": 0.0,
            "adjustment_applied_amount": 0.0,
            "adjustment_exchange_rate": 1.0,
        }
        resp = await client.post("/ar_records/submit", json=record)
        return idx, resp.status_code, resp.json()


async def test_concurrent_submissions():
    """Test concurrent submissions with the same customer_id."""
    customer_id = "CUST-CONCURRENT-001"

    # Fire 5 concurrent requests
    tasks = [submit_record_async(customer_id, i) for i in range(5)]
    results = await asyncio.gather(*tasks)

    print(f"=== Concurrent Submission Results for {customer_id} ===")
    workflow_ids = set()
    for idx, status_code, data in results:
        wf_id = data.get("workflow_id")
        workflow_ids.add(wf_id)
        print(f"Request {idx}: Status {status_code}, Workflow {wf_id[:8]}...")

    print(f"\nUnique workflow IDs: {len(workflow_ids)}")
    if len(workflow_ids) == 1:
        print("✅ Concurrent idempotency verified - only one workflow created")
    else:
        print("❌ Multiple workflows created - race condition detected")


# Run the concurrent test
await test_concurrent_submissions()

=== Concurrent Submission Results for CUST-CONCURRENT-001 ===
Request 0: Status 201, Workflow b4decbb5...
Request 1: Status 201, Workflow b4decbb5...
Request 2: Status 201, Workflow b4decbb5...
Request 3: Status 201, Workflow b4decbb5...
Request 4: Status 201, Workflow b4decbb5...

Unique workflow IDs: 1
✅ Concurrent idempotency verified - only one workflow created


---
## 15. Workflow Lifecycle Summary

Display a complete lifecycle view of a single workflow.

In [55]:
# Create a new workflow and track its lifecycle
lifecycle_record = {
    "customer_id": "CUST-LIFECYCLE-001",
    "customer_name": "Lifecycle Test Customer",
    "customer_balance": 2500.00,
    "invoice_total": 3000.00,
    "invoice_applied_amount": 1000.00,
    "invoice_exchange_rate": 1.0,
    "payment_total": 1500.00,
    "payment_applied_amount": 1000.00,
    "payment_exchange_rate": 1.0,
    "credit_total": 200.00,
    "credit_applied_amount": 100.00,
    "credit_exchange_rate": 1.0,
    "adjustment_total": 50.00,
    "adjustment_applied_amount": 25.00,
    "adjustment_exchange_rate": 1.0,
}

print("=== Creating Workflow ===")
resp = client.post("/ar_records/submit", json=lifecycle_record)
result = resp.json()
lifecycle_workflow_id = result.get("workflow_id")
print(f"Workflow ID: {lifecycle_workflow_id}")
print(f"Initial Status: {result.get('workflow_status')}")

# Poll for completion
print("\n=== Tracking Progress ===")
for i in range(15):
    time.sleep(1)
    resp = client.get(f"/ar_records/workflow/{lifecycle_workflow_id}")
    wf = resp.json()["workflow"]
    print(f"  [{i + 1}s] Status: {wf['status']} | Stage: {wf['current_stage']}")
    if wf["status"] in ["COMPLETED", "FAILED"]:
        break

# Final state
print("\n=== Final Workflow State ===")
resp = client.get(f"/ar_records/workflow/{lifecycle_workflow_id}")
final = resp.json()
print(f"Final Status: {final['workflow']['status']}")
print(f"Total Stages: {len(final['stages'])}")
print("\nStage Execution Summary:")
for stage in final["stages"]:
    status_emoji = (
        "✅"
        if stage["status"] == "COMPLETED"
        else "❌"
        if stage["status"] == "FAILED"
        else "🔄"
    )
    print(
        f"  {status_emoji} {stage['stage_name']}: {stage['status']} (retries: {stage['retry_count']})"
    )

=== Creating Workflow ===
Workflow ID: 5d6d8dd9-6023-4b37-80bb-1242cd40f929
Initial Status: PENDING

=== Tracking Progress ===
  [1s] Status: COMPLETED | Stage: REPORTING

=== Final Workflow State ===
Final Status: COMPLETED
Total Stages: 7

Stage Execution Summary:
  ✅ INGESTION: COMPLETED (retries: 0)
  ✅ NORMALIZATION: COMPLETED (retries: 0)
  ✅ BALANCE_COMPUTE: COMPLETED (retries: 0)
  ✅ RECONCILIATION: COMPLETED (retries: 0)
  ✅ VALIDATION_RULES: COMPLETED (retries: 0)
  ✅ VERDICT_GENERATION: COMPLETED (retries: 0)
  ✅ REPORTING: COMPLETED (retries: 0)


---
## Cleanup

Close the HTTP client.

In [56]:
client.close()
print("✅ HTTP client closed")
print("\n=== All tests completed ===")

✅ HTTP client closed

=== All tests completed ===


# AR Reconciliation API - Endpoint Testing

## Setup
```
uv sync
uv sync --extra dev
uv run alembic revision --autogenerate -m "your_message_here"
uv run alembic upgrade head
uv run python main.py
```
```
CREATE OR REPLACE VIEW view_name AS
SELECT
    t1.id,
    t1.name,
    t2.status,
    t2.created_at
FROM table1 t1
LEFT JOIN table2 t2
    ON t1.id = t2.table1_id;
```


In [57]:
import time

import httpx

BASE_URL = "http://localhost:8000"
client = httpx.Client(base_url=BASE_URL, timeout=30)

## 1. Health Check
`GET /health` - Verify the server is running.

In [58]:
resp = client.get("/health")
print(f"Status: {resp.status_code}")
resp.json()

Status: 200


{'message': None,
 'status': 'healthy',
 'version': '1.0.0',
 'timestamp': '2026-06-01T11:04:32.012787+05:30',
 'services': {'database': 'ok'}}

## 2. Submit a Single Record
`POST /submit` - Submit one AR record for processing. Returns a workflow ID.

In [59]:
record = {
    "customer_id": "abcss33",
    "customer_name": "Customer fsssfs",
    "customer_balance": 3486.58,
    "invoice_total": 3892.38,
    "invoice_applied_amount": 241.57,
    "invoice_exchange_rate": 1.1371,
    "payment_total": 3316.88,
    "payment_applied_amount": 2375.22,
    "payment_exchange_rate": 0.8873,
    "credit_total": 910.95,
    "credit_applied_amount": 676.97,
    "credit_exchange_rate": 1.0237,
    "adjustment_total": 1310.49,
    "adjustment_applied_amount": 944.51,
    "adjustment_exchange_rate": 1.1211,
}

resp = client.post("/ar_records/submit", json=record)
print(f"Status: {resp.status_code}")
resp.json()

Status: 201


{'message': 'Record submitted successfully.',
 'customer_id': 'abcss33',
 'workflow_id': 'd2757cde-a26a-4cc8-9b15-7d854943daf7',
 'workflow_status': 'PENDING',
 'workflow_current_stage': 'INGESTION'}

## 3. Submit Duplicate (Idempotency Check)
Submitting the same `customer_id` again should return the existing workflow, not create a new one.

In [60]:
# Submit the same record again - should be idempotent
resp = client.post("/ar_records/submit", json=record)
print(f"Status: {resp.status_code}")
resp.json()

Status: 201


{'message': 'Duplicate submission detected. Returning existing workflow details. Use the update endpoint to modify customer data.',
 'customer_id': 'abcss33',
 'workflow_id': 'd2757cde-a26a-4cc8-9b15-7d854943daf7',
 'workflow_status': 'RUNNING',
 'workflow_current_stage': 'NORMALIZATION'}

# TEST pipeline

In [61]:
resp = client.post(
    "/ar_pipeline/run",
    json={
        "workflow_id": "b99c2148-e790-456d-a510-51e9bf55ebcc",
        "customer_id": "abcss33",
    },
)

print(resp.status_code)
print(resp.json())

404
{'detail': 'Not Found'}


## 4. Get Workflow Status
`GET /workflow/{id}` - Check the status and stage details of a workflow.

In [62]:
# Wait a moment for background processing to complete
time.sleep(2)

resp = client.get(f"/workflow/{workflow_id}")
print(f"Status: {resp.status_code}")
workflow_detail = resp.json()
print(f"Workflow status: {workflow_detail['workflow']['status']}")
print(f"Current stage: {workflow_detail['workflow']['current_stage']}")
print(f"Number of stages recorded: {len(workflow_detail['stages'])}")
workflow_detail

Status: 404


KeyError: 'workflow'

## 5. Update a Record
`PUT /submit/{customer_id}` - Update an existing record and reprocess from scratch.

In [ ]:
# Update the record with a different balance
updated_record = record.copy()
updated_record["customer_balance"] = 5000.00
updated_record["invoice_total"] = 5500.00

resp = client.put("/submit/CUST-0001", json=updated_record)
print(f"Status: {resp.status_code}")
resp.json()

## 6. Bulk Upload CSV
`POST /bulk-upload` - Upload the CSV file to process multiple records at once.

In [ ]:
# Upload the CSV file for bulk processing
with open("../data/erp_export.csv", "rb") as f:
    resp = client.post(
        "/bulk-upload", files={"file": ("erp_export.csv", f, "text/csv")}
    )

print(f"Status: {resp.status_code}")
bulk_result = resp.json()
print(f"Total records: {bulk_result['total_records']}")
print(f"Submitted: {bulk_result['submitted']}")
print(f"Duplicates: {bulk_result['duplicates']}")
print(f"Workflow IDs (first 5): {bulk_result['workflow_ids'][:5]}")
bulk_result

## 7. List All Workflows
`GET /workflows` - List workflows with optional status filter and pagination.

In [ ]:
# Wait for background processing
time.sleep(3)

# List all workflows
resp = client.get("/workflows")
print(f"Status: {resp.status_code}")
workflows = resp.json()
print(f"Total workflows returned: {len(workflows)}")
# Show first 3
for w in workflows[:3]:
    print(
        f"  {w['id']} | {w['customer_id']} | {w['status']} | stage: {w['current_stage']}"
    )

In [ ]:
# Filter workflows by status
resp = client.get("/workflows", params={"status": "COMPLETED", "limit": 5})
print(f"Completed workflows: {resp.status_code}")
print(f"Count: {len(resp.json())}")

resp = client.get("/workflows", params={"status": "FAILED", "limit": 5})
print(f"\nFailed workflows: {resp.status_code}")
failed_workflows = resp.json()
print(f"Count: {len(failed_workflows)}")
for w in failed_workflows[:3]:
    print(f"  {w['id']} | {w['customer_id']} | retry_count: {w['retry_count']}")

## 8. Resume a Failed Workflow
`POST /resume/{id}` - Resume a workflow that failed (retries from the last successful stage).

In [ ]:
# Find a failed workflow to resume
resp = client.get("/workflows", params={"status": "FAILED", "limit": 1})
failed = resp.json()

if failed:
    failed_id = failed[0]["id"]
    print(f"Resuming workflow: {failed_id}")
    resp = client.post(f"/resume/{failed_id}")
    print(f"Status: {resp.status_code}")
    print(resp.json())

    # Check status after a moment
    time.sleep(2)
    resp = client.get(f"/workflow/{failed_id}")
    print(f"\nAfter resume - Status: {resp.json()['workflow']['status']}")
else:
    print("No failed workflows to resume (all succeeded!)")
    print(
        "Note: The system has a 20% random failure rate, so run bulk-upload again if needed"
    )

## 9. Dashboard Stats
`GET /stats` - Get aggregate statistics about all workflows.

In [ ]:
resp = client.get("/stats")
print(f"Status: {resp.status_code}")
stats = resp.json()
print(f"Total workflows: {stats['total_workflows']}")
print(f"Completed: {stats['completed']}")
print(f"Failed: {stats['failed']}")
print(f"Pending: {stats['pending']}")
print(f"Running: {stats['running']}")
print(f"Stale: {stats['stale']}")
print(f"Routing decisions: {stats['decisions']}")

## 10. Enhanced Workflow List
`GET /workflows/enhanced` - List workflows with staleness flag and last error message.

In [ ]:
# Enhanced list with error details
resp = client.get("/workflows/enhanced", params={"limit": 5})
print(f"Status: {resp.status_code}")
enhanced = resp.json()
for w in enhanced[:5]:
    stale_flag = " [STALE]" if w.get("is_stale") else ""
    error = f" | error: {w['last_error']}" if w.get("last_error") else ""
    print(f"  {w['customer_id']} | {w['status']}{stale_flag}{error}")

## 11. Export Results as CSV
`GET /export` - Download completed workflow results as a CSV file.

In [ ]:
# Export completed results as CSV
resp = client.get("/export")
print(f"Status: {resp.status_code}")
print(f"Content-Type: {resp.headers.get('content-type')}")
print("\nCSV Preview (first 500 chars):")
print(resp.text[:500])

## 12. Error Handling - Invalid Requests
Test that the API returns proper error responses for bad input.

In [ ]:
# Test 404 - workflow not found
resp = client.get("/workflow/nonexistent-id-12345")
print(f"GET /workflow/bad-id -> {resp.status_code} (expected 404)")
print(f"  Response: {resp.json()}")

# Test 404 - resume non-existent workflow
resp = client.post("/resume/nonexistent-id-12345")
print(f"\nPOST /resume/bad-id -> {resp.status_code} (expected 404)")
print(f"  Response: {resp.json()}")

# Test 400 - upload non-CSV file
resp = client.post(
    "/bulk-upload", files={"file": ("test.txt", b"not a csv", "text/plain")}
)
print(f"\nPOST /bulk-upload with .txt -> {resp.status_code} (expected 400)")
print(f"  Response: {resp.json()}")

# Test 422 - missing required field
resp = client.post("/submit", json={})
print(f"\nPOST /submit with empty body -> {resp.status_code} (expected 422)")
print(f"  Response: {resp.json()['detail'][0]['msg']}")

## 13. Resume All Failed Workflows
Loop through all failed workflows and attempt to resume them.

In [ ]:
# Resume all failed workflows
resp = client.get("/workflows", params={"status": "FAILED", "limit": 200})
failed = resp.json()
print(f"Found {len(failed)} failed workflows. Resuming...")

for w in failed:
    r = client.post(f"/resume/{w['id']}")
    print(f"  {w['customer_id']}: {r.json()['message']}")

# Wait and check final stats
time.sleep(5)
resp = client.get("/stats")
stats = resp.json()
print("\n--- Final Stats ---")
print(f"Completed: {stats['completed']}/{stats['total_workflows']}")
print(f"Still failed: {stats['failed']}")
print(f"Decisions: {stats['decisions']}")

---

## 14. Concurrent Duplicate Submission
Fire multiple requests for the same customer_id at once - only one workflow should be created.


In [ ]:
import concurrent.futures

# Use a unique customer ID for this test
test_customer_id = "RACE-COND-TEST-001"
race_record = {
    "customer_id": test_customer_id,
    "customer_name": "Race Condition Test",
    "invoice_total": 1000.00,
    "payment_total": 1000.00,
}


def submit_record(record):
    """Submit in a separate thread to simulate concurrency."""
    c = httpx.Client(base_url=BASE_URL, timeout=30)
    return c.post("/submit", json=record).json()


# Fire 5 concurrent requests for the same customer
with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
    futures = [executor.submit(submit_record, race_record) for _ in range(5)]
    results = [f.result() for f in concurrent.futures.as_completed(futures)]

# All should return the same workflow_id (no duplicates created)
workflow_ids = set(r["workflow_id"] for r in results)
statuses = [r["status"] for r in results]

print("Concurrent submissions: 5")
print(f"Unique workflow IDs returned: {len(workflow_ids)} (expected: 1)")
print(f"Workflow ID: {workflow_ids.pop()}")
print(f"Statuses: {statuses}")
assert len(workflow_ids) == 0, "ERROR: Multiple workflows created for same customer!"
print("\n[ok] Race condition handled correctly - no duplicate workflows")

## 15. ACID / Atomicity Check
Submit same record twice, confirm idempotency key dedup works and stages are persisted atomically.


In [ ]:
# Submit the same record data twice via the API - second should be idempotent
acid_record = {
    "customer_id": "ACID-TEST-001",
    "customer_name": "ACID Test Customer",
    "invoice_total": 2500.00,
    "payment_total": 2500.00,
}

# First submission
resp1 = client.post("/submit", json=acid_record)
result1 = resp1.json()
print(f"1st submit: status={result1['status']}, workflow_id={result1['workflow_id']}")

time.sleep(2)

# Second submission (same data) - should return DUPLICATE via idempotency
resp2 = client.post("/submit", json=acid_record)
result2 = resp2.json()
print(f"2nd submit: status={result2['status']}, workflow_id={result2['workflow_id']}")

# Verify same workflow is returned
assert result1["workflow_id"] == result2["workflow_id"], (
    "ERROR: Different workflow IDs!"
)
print("\n[ok] Idempotency verified - same workflow returned for duplicate data")

# Verify workflow completed with all stages atomically persisted
resp = client.get(f"/workflow/{result1['workflow_id']}")
detail = resp.json()
print(f"\nWorkflow status: {detail['workflow']['status']}")
print(f"Stages recorded: {len(detail['stages'])}")
for stage in detail["stages"]:
    if stage["status"] == "SUCCESS":
        print(f"  [+] {stage['stage_name']}: {stage['status']}")
    else:
        print(
            f"  X {stage['stage_name']}: {stage['status']} - {stage.get('error_message', '')}"
        )

## 16. Retry Exhaustion
Confirm workflows end up in FAILED (not stuck in RUNNING) when retries are exhausted.


In [ ]:
# Submit multiple records to exercise the retry mechanism
retry_records = [
    {
        "customer_id": f"RETRY-TEST-{i:03d}",
        "customer_name": f"Retry Test {i}",
        "invoice_total": 1000.0 + i,
        "payment_total": 900.0 + i,
    }
    for i in range(10)
]

submitted_ids = []
for rec in retry_records:
    resp = client.post("/submit", json=rec)
    result = resp.json()
    submitted_ids.append(result["workflow_id"])

print(f"Submitted {len(submitted_ids)} records for retry testing")
print("Waiting for background processing (with 20% failure rate)...")
time.sleep(5)

# Check that NO workflow is stuck in RUNNING state
resp = client.get("/workflows", params={"status": "RUNNING", "limit": 200})
running = resp.json()
print(f"\nWorkflows stuck in RUNNING: {len(running)} (expected: 0)")

# Check retry counts on failed workflows
resp = client.get("/workflows", params={"status": "FAILED", "limit": 200})
failed = resp.json()
print(f"Workflows that FAILED after retries: {len(failed)}")
for w in failed[:5]:
    print(
        f"  {w['customer_id']} | retry_count={w['retry_count']} | stage={w['current_stage']}"
    )

resp = client.get("/workflows", params={"status": "COMPLETED", "limit": 200})
completed = resp.json()
print(f"Workflows COMPLETED: {len(completed)}")

if len(running) == 0:
    print("\n[ok] No workflows stuck in RUNNING - failure handling works correctly")
else:
    print("\nX WARNING: Some workflows stuck in RUNNING state!")

## 17. Resume from Checkpoint
Resume a failed workflow and confirm it picks up from the last successful stage.


In [ ]:
# Find a failed workflow that has some successful stages
resp = client.get("/workflows", params={"status": "FAILED", "limit": 10})
failed = resp.json()

resumed_wf = None
for w in failed:
    detail_resp = client.get(f"/workflow/{w['id']}")
    detail = detail_resp.json()
    successful_stages = [s for s in detail["stages"] if s["status"] == "SUCCESS"]
    failed_stages = [s for s in detail["stages"] if s["status"] == "FAILED"]

    if successful_stages:
        resumed_wf = w
        print(f"Found workflow with partial progress: {w['id']}")
        print(f"  Customer: {w['customer_id']}")
        print(f"  Successful stages: {[s['stage_name'] for s in successful_stages]}")
        print(f"  Failed at: {w['current_stage']} (retry_count={w['retry_count']})")
        break

if resumed_wf:
    # Resume it
    resp = client.post(f"/resume/{resumed_wf['id']}")
    print(f"\nResume response: {resp.json()['message']}")
    time.sleep(3)

    # Check if it progressed further
    resp = client.get(f"/workflow/{resumed_wf['id']}")
    detail = resp.json()
    print(f"After resume - status: {detail['workflow']['status']}")
    print("Stage history:")
    for s in detail["stages"]:
        marker = "[+]" if s["status"] == "SUCCESS" else "X"
        print(f"  {marker} {s['stage_name']}: {s['status']}")
    print("\n[ok] Resume from checkpoint works - state was reconstructed correctly")
else:
    print("No failed workflows with partial progress found (all may have completed)")
    print("This is expected when failure rate is low or all retries succeeded")

## 18. DB Consistency Audit
Check that COMPLETED workflows have all 4 stages, FAILED ones have recorded errors, none stuck in RUNNING.


In [ ]:
# Audit all workflows for DB consistency
resp = client.get("/workflows", params={"limit": 200})
all_workflows = resp.json()

inconsistent = []
for w in all_workflows:
    detail_resp = client.get(f"/workflow/{w['id']}")
    detail = detail_resp.json()
    stages = detail["stages"]
    successful = [s for s in stages if s["status"] == "SUCCESS"]

    if w["status"] == "COMPLETED":
        # Completed workflows must have all 4 stages successful
        if len(successful) < 4:
            inconsistent.append(
                f"COMPLETED but only {len(successful)} successful stages: {w['id']}"
            )
    elif w["status"] == "FAILED":
        # Failed workflows must have at least one FAILED stage recorded
        failed_stages = [s for s in stages if s["status"] == "FAILED"]
        if not failed_stages:
            inconsistent.append(f"FAILED but no failed stages recorded: {w['id']}")
    elif w["status"] == "RUNNING":
        # No workflow should be stuck in RUNNING (pipeline guard catches this)
        inconsistent.append(f"Stuck in RUNNING: {w['id']}")

print(f"Total workflows audited: {len(all_workflows)}")
print(f"Inconsistencies found: {len(inconsistent)}")

if inconsistent:
    for issue in inconsistent:
        print(f"  X {issue}")
else:
    print("\n[ok] All workflows are in a consistent state - ACID properties maintained")

# Summary
statuses = {}
for w in all_workflows:
    statuses[w["status"]] = statuses.get(w["status"], 0) + 1
print(f"\nStatus distribution: {statuses}")

## 19. State Machine Checks
- Can't resume a COMPLETED workflow
- Non-existent workflow returns 404
- Only valid statuses exist in DB


In [ ]:
# Test: Cannot resume a COMPLETED workflow
resp = client.get("/workflows", params={"status": "COMPLETED", "limit": 1})
completed = resp.json()

if completed:
    wf_id = completed[0]["id"]
    resp = client.post(f"/resume/{wf_id}")
    result = resp.json()
    print(f"Resume completed workflow: {result['message']}")
    assert "already completed" in result["message"].lower(), (
        "Should indicate already completed"
    )
    print("[ok] Completed workflow correctly rejects resume")
else:
    print("No completed workflows to test (run bulk upload first)")

# Test: Resume non-existent workflow returns 404
resp = client.post("/resume/does-not-exist-xyz")
print(f"\nResume non-existent workflow: {resp.status_code} (expected 404)")
assert resp.status_code == 404
print("[ok] Non-existent workflow correctly returns 404")

# Test: Check that workflow statuses are only valid values
resp = client.get("/workflows", params={"limit": 200})
all_wf = resp.json()
valid_statuses = {"PENDING", "RUNNING", "COMPLETED", "FAILED"}
invalid = [w for w in all_wf if w["status"] not in valid_statuses]
print(f"\nWorkflows with invalid status: {len(invalid)} (expected: 0)")
assert len(invalid) == 0, f"Found invalid statuses: {[w['status'] for w in invalid]}"
print("[ok] All workflow statuses are valid state machine values")